# iot-detector demo

This notebook shows how to use the package. It uses **synthetic (invented) data**, so it runs without the CICIoT2023 files. The numbers say nothing about real traffic.

Run it from the repo root after `pip install -e ".[dev]"`.

In [ ]:
import warnings
import numpy as np, pandas as pd
import iot_detector
from iot_detector import IoTDetector, FEATURES
from iot_detector.demo import make_synthetic
from iot_detector.data import split_by_file
from iot_detector.model import evaluate, calibration_table
print(iot_detector.__version__)

## 1. Data and file-grouped split
Whole files go to train or test, as in Part II.

In [ ]:
df = make_synthetic(n=4000, seed=0)
train, test = split_by_file(df)
print(len(train), len(test), 'files in both:', set(train.source_file) & set(test.source_file))

## 2. Train (fixed seed 42)
Only the 39 `FEATURES` are used. `Label`, `y` and `source_file` are not inputs.

In [ ]:
det = IoTDetector(kind='rf', n_estimators=100).fit(train[FEATURES], train['y'])
det.meta_['versions']

## 3. Predict

In [ ]:
res = det.predict(test[FEATURES].head(5))
res

## 4. Bad inputs: warning or refusal

In [ ]:
bad = test[FEATURES].head(4).copy()
bad.loc[bad.index[0], 'Rate'] = np.inf          # infinite -> warn, repair
bad.loc[bad.index[1], 'Header_Length'] = -5     # out of range -> warn, repair
bad.loc[bad.index[2], 'Protocol Type'] = 99     # unseen category -> refuse
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    out = det.predict(bad)
for x in w: print('WARNING:', x.message)
out

In [ ]:
try:
    det.predict(test[FEATURES].drop(columns=['Rate']).head(3))
except ValueError as e:
    print('REFUSED:', e)

## 5. Check on held-out files, and honest uncertainty
The output is a **probability**, not a confidence interval. The table checks whether the probabilities match what happened.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    p = det.predict_proba(test[FEATURES])
print(evaluate(test['y'], p))
calibration_table(test['y'], p)

## 6. Save and load
Saved models store seed and library versions.

In [ ]:
det.save('demo_model.joblib')
again = IoTDetector.load('demo_model.joblib')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    same = np.allclose(det.predict_proba(test[FEATURES]), again.predict_proba(test[FEATURES]), equal_nan=True)
print('same predictions after reload:', same)
import os; os.remove('demo_model.joblib')